Copyright (C) 2026 The AiQuokkaWannabe Authors <https://github.com/ai-quokka-wannabe/iso-lab>

This program is free software: you can redistribute it and/or modify it under the terms of the GNU General Public
License as published by the Free Software Foundation, either version 3 of the License, or (at your option) any later
version.

This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY; without even the implied
warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the GNU General Public License for more details.

You should have received a copy of the GNU General Public License along with this program. If not, see
<https://www.gnu.org/licenses/>.

# H10, First Test: Does Phase Buy Memory?

*A hunch*, tested on a toy problem before any Program exists. H10 says a quantum-like brain, whose states and
connection strengths are complex numbers, may do what a real-valued one cannot. This notebook asks the smallest
question that could refute it: **at the same memory budget, does a complex-valued (phasor) associative memory hold
more retrievable information than a real-valued Hopfield network?**

## Pre-registration (written and committed before the first run)

- **Real network**: N = 256 units of state +1 or -1, Hebbian weights W = (1/N) sum of outer products, zero
  diagonal, asynchronous updates in a seeded random order, sign of the local field (a zero field keeps the state),
  at most 30 sweeps or until no unit changes.
- **Phasor network**: units take one of K = 4 phases, exp(2 pi i k / 4). Hermitian Hebbian weights
  W = (1/N) sum of xi conj(xi)^T, zero diagonal; a unit takes the phase nearest the argument of its local field
  (a zero field keeps the state); the same update schedule. The update follows the multistate complex-valued
  memories of Noest 1988 and Jankowski, Lozowski and Zurada 1996, cited for the idea only - neither was read
  in full here, and no number is taken from them.
- **Same memory**: the phasor network has round(256 / sqrt 2) = 181 units, so its 181 x 181 complex weights are
  65 522 real numbers against the real network's 65 536.
- **Load**: P = 2, 4, ..., 60 stored patterns, uniform at random. For each P, up to 20 of the stored patterns are
  cued, each with 10% of its units (rounded) set to a different random state.
- **Success**: the overlap |sum conj(xi) z| / N reaches 0.98 at the end (a global phase or sign is allowed).
- **The measure**: retrievable information at load P is P x (bits per pattern) x (fraction of cues recalled);
  bits per pattern are 256 for the real network and 181 x log2 4 = 362 for the phasor one. Per seed, the score is
  the maximum over P, divided by the real numbers stored.
- **Seeds**: 20 (0 to 19), each fixing patterns, cues and update orders, paired between the two networks.
- **Statistic**: the mean paired difference, phasor minus real, with a 95% percentile bootstrap interval
  (10 000 resamples, generator seed 20261013).
- **Refutation**: this test refutes H10's memory claim unless the interval's lower bound is above zero. An interval
  that includes zero counts as *no better*, and so as a refutation at this level.
- **Control (the implementation check)**: the phasor code, in complex arithmetic, run with K = 2 and N = 256 on
  the same seeds as the real network, must score like it - the paired interval must include zero. If it does not,
  the run is void, not a result.
- **Cost, reported beside the score**: real multiply-adds per sweep, N^2 for the real network and 4 x 181^2 for the
  phasor one, since one complex multiply-add costs four real ones. The lean rule decides what a win is worth.
- **What this does not show**: anything about a worm. It is a toy memory, not a Program, and no benchmark exists
  yet; a pass earns H10 a second test, not a place in a brain.

In [1]:
import numpy as np

N_REAL = 256
K = 4
N_PHASOR = round(N_REAL / np.sqrt(2))
NOISE = 0.10
MAX_SWEEPS = 30
LOADS = np.arange(2, 61, 2)
CUES_PER_LOAD = 20
SEEDS = range(20)
THRESHOLD = 0.98
BOOT = 10_000
BOOT_SEED = 20261013
print(np.__version__, N_PHASOR, N_PHASOR**2 * 2, N_REAL**2)

2.5.4 181 65522 65536


In [2]:
def recall(W, cue, states, rng):
    """Asynchronous recall: each unit takes the state nearest its local field; a zero field keeps it."""
    z = cue.copy()
    n = len(z)
    for _ in range(MAX_SWEEPS):
        changed = False
        for j in rng.permutation(n):
            h = W[j] @ z
            if abs(h) < 1e-12:
                continue
            new = states[np.argmax((np.conj(states) * h).real)]
            if new != z[j]:
                z[j] = new
                changed = True
        if not changed:
            break
    return z


def capacity(n, k, seed, real=False):
    """Fraction of cues recalled at each load, for a k-state network of n units; real=True is the real network."""
    rng = np.random.default_rng(seed)
    states = np.exp(2j * np.pi * np.arange(k) / k)
    if real:
        states = states.real
    flips = round(NOISE * n)
    recalled = []
    for p in LOADS:
        idx = rng.integers(0, k, size=(p, n))
        xi = states[idx]
        W = (xi.T @ np.conj(xi)) / n
        np.fill_diagonal(W, 0)
        ok = 0
        cues = min(p, CUES_PER_LOAD)
        for m in rng.choice(p, size=cues, replace=False):
            cue = xi[m].copy()
            for j in rng.choice(n, size=flips, replace=False):
                others = [s for s in states if s != cue[j]]
                cue[j] = others[rng.integers(0, len(others))]
            z = recall(W, cue, states, rng)
            ok += abs(np.vdot(xi[m], z)) / n >= THRESHOLD
        recalled.append(ok / cues)
    return np.array(recalled)


def score(recalled, bits_per_pattern, reals):
    return np.max(LOADS * bits_per_pattern * recalled) / reals

In [3]:
real = np.array([capacity(N_REAL, 2, s, real=True) for s in SEEDS])
phasor = np.array([capacity(N_PHASOR, K, s) for s in SEEDS])
control = np.array([capacity(N_REAL, 2, s) for s in SEEDS])

s_real = np.array([score(r, N_REAL, N_REAL**2) for r in real])
s_phasor = np.array([score(r, N_PHASOR * np.log2(K), 2 * N_PHASOR**2) for r in phasor])
s_control = np.array([score(r, N_REAL, N_REAL**2) for r in control])

In [4]:
def paired_ci(a, b):
    d = a - b
    rng = np.random.default_rng(BOOT_SEED)
    means = rng.choice(d, size=(BOOT, len(d)), replace=True).mean(axis=1)
    return d.mean(), np.percentile(means, 2.5), np.percentile(means, 97.5)


print("bits per real weight, mean over seeds")
print(f"  real   {s_real.mean():.4f}  (sd {s_real.std(ddof=1):.4f})")
print(f"  phasor {s_phasor.mean():.4f}  (sd {s_phasor.std(ddof=1):.4f})")
print(f"  control (complex code, K = 2) {s_control.mean():.4f}")
print("phasor - real: mean %.4f, 95%% CI [%.4f, %.4f]" % paired_ci(s_phasor, s_real))
print("control - real: mean %.4f, 95%% CI [%.4f, %.4f]" % paired_ci(s_control, s_real))
for name, r in (("real", real), ("phasor", phasor)):
    print(name, "fraction recalled by load:", np.round(r.mean(axis=0), 2).tolist())
print("multiply-adds per sweep: real", N_REAL**2, "phasor", 4 * N_PHASOR**2)

bits per real weight, mean over seeds
  real   0.1119  (sd 0.0076)
  phasor 0.1149  (sd 0.0061)
  control (complex code, K = 2) 0.1119
phasor - real: mean 0.0030, 95% CI [-0.0010, 0.0068]
control - real: mean 0.0000, 95% CI [0.0000, 0.0000]
real fraction recalled by load: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.99, 0.97, 0.94, 0.91, 0.87, 0.83, 0.72, 0.6, 0.54, 0.44, 0.35, 0.24, 0.22, 0.14, 0.09, 0.06, 0.05, 0.03, 0.01, 0.02]
phasor fraction recalled by load: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.98, 0.96, 0.9, 0.77, 0.63, 0.5, 0.34, 0.2, 0.14, 0.06, 0.02, 0.01, 0.01, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
multiply-adds per sweep: real 65536 phasor 131044
